<a href="https://colab.research.google.com/github/badwbal/ADF/blob/main/HMLR_DS_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
from google.colab import files
uploaded = files.upload()

Saving anonymised 1.pdf to anonymised 1.pdf


In [26]:
from google.colab import files
uploaded = files.upload()

Saving analysis_report.txt to analysis_report.txt


In [28]:
# Install system packages
!apt-get install poppler-utils tesseract-ocr -y

# Install python libraries
!pip install pytesseract pdf2image

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [29]:
import os
os.listdir()

['.config',
 'classification_results.csv',
 'analysis_report.txt',
 'C:\\Users\\new\\Documents\\1 Roles_applied\\HM Land Registry\\Task\\Neethu',
 'anonymised 1.pdf',
 'sample_data']

In [30]:
import pytesseract
from pdf2image import convert_from_path
import re
import csv
import os

In [31]:
def extract_text_with_ocr(pdf_path):
    images = convert_from_path(pdf_path)
    all_text = []

    for image in images:
        text = pytesseract.image_to_string(image)
        all_text.append(text)

    return all_text

In [33]:
def classify_page(text, page_num):

    if not text or len(text.strip()) < 10:
        return "BLANK_PAGE"

    if page_num == 1:
        return "NUMERIC_INDEX_PAGE"

    if "the development hereby permitted" in text.lower():
        return "DECISION_NOTICE_CONDITIONS"

    if "notice of approval" in text.lower():
        return "APPROVAL_OF_DETAILS"

    return "OTHER"

In [35]:
def extract_application_numbers(text):

    return list(set(re.findall(r'P/\d{2}/\d{4}', text)))

In [36]:
def extract_applicant_name(text):

    if "Mrs AM Stephens" in text:
        return "Mrs AM Stephens"

    if "Mr M Dale" in text:
        return "Mr M Dale"

    return "Not found"

In [37]:
pdf_file = "anonymised 1.pdf"

pages = extract_text_with_ocr(pdf_file)

results = []

for i, text in enumerate(pages, 1):

    results.append({
        'page': i,
        'type': classify_page(text, i),
        'apps': ', '.join(extract_application_numbers(text)),
        'applicant': extract_applicant_name(text)
    })

In [38]:
with open('classification_results.csv', 'w', newline='') as f:

    w = csv.writer(f)

    w.writerow(['Page', 'Type', 'Application Numbers', 'Applicant'])

    for r in results:
        w.writerow([r['page'], r['type'], r['apps'], r['applicant']])

In [39]:
for r in results:
    print(f"Page {r['page']}: {r['type']} | {r['apps']} | {r['applicant']}")

Page 1: NUMERIC_INDEX_PAGE |  | Not found
Page 2: DECISION_NOTICE_CONDITIONS | P/00/0759 | Mr M Dale
Page 3: DECISION_NOTICE_CONDITIONS |  | Not found
Page 4: APPROVAL_OF_DETAILS | P/96/0900, P/98/0964 | Mrs AM Stephens


In [40]:
import pandas as pd
pd.read_csv("classification_results.csv")

,Page,Type,Application Numbers,Applicant
0,1,NUMERIC_INDEX_PAGE,NaN,Not found
1,2,DECISION_NOTICE_CONDITIONS,P/00/0759,Mr M Dale
2,3,DECISION_NOTICE_CONDITIONS,NaN,Not found
3,4,APPROVAL_OF_DETAILS,"P/96/0900, P/98/0964",Mrs AM Stephens


In [42]:
files.download("classification_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>